# Experiment 08 (served data) — EfficientNetB0 + SMOTE + XAI / AUC / Grad-CAM

**Purpose:** identical model, training, and evaluation pipeline to `exp08`, but the
data is loaded directly from the pre-computed `.npy` files instead of downloading
APTOS and preprocessing in-notebook.

**What changed vs. exp08:**
1. No Kaggle download, no in-notebook preprocessing, no in-notebook split, no in-notebook SMOTE.
2. The six served arrays (`X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test`) are
   loaded from `data/preprocessed/`. These were produced by the `dr_grading`
   pipeline: resize → LAB-CLAHE → median denoise → `float32 [0, 1]`, then **SMOTE on the
   training split only**.
3. Pixels are rescaled once from `[0, 1]` → `[0, 255]` after loading, because Keras
   `EfficientNetB0` and the augmentation layers expect the `[0, 255]` range. Everything
   **from augmentation onwards is unchanged** relative to exp08.


## 1. Environment Setup

In [ ]:
# Optional: install runtime deps. TensorFlow is assumed present (e.g. Colab GPU runtime).
# imbalanced-learn / kaggle are intentionally NOT needed — SMOTE and download happen upstream.
!pip install -q opencv-python scikit-learn matplotlib pandas tqdm


In [ ]:
# Optional Google Colab Drive mount. Safe no-op when running locally.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Not running in Colab (or Drive already mounted); using local paths.", exc)


Mounted at /content/drive


In [ ]:
%cd drive/MyDrive

/content/drive/MyDrive


In [ ]:
# Reproducibility: seed Python, NumPy, and TensorFlow so runs are comparable.
import os
import random

import numpy as np
import tensorflow as tf

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
# Full op-determinism (tf.config.experimental.enable_op_determinism) is left OFF:
# some GPU augmentation kernels have no deterministic implementation and would error.
print("Seeds set to", SEED)


Seeds set to 42


In [ ]:
from pathlib import Path

# ----- Core constants (must match how the served arrays were produced) -----
IMG_SIZE = 224
NUM_CLASSES = 5
BATCH_SIZE = 32

# ----- Where the served preprocessed + SMOTE-balanced .npy files live -----
# The pipeline writes to <project>/data/preprocessed. Override PREPROCESSED_DIR
# directly if your data sits elsewhere (e.g. a Colab Drive path).
_CANDIDATE_DIRS = [
    Path("data/preprocessed"),
    Path("capstone-project/data/preprocessed"),
    Path("/content/drive/MyDrive/dr_severity_aptos/data/preprocessed"),
]
PREPROCESSED_DIR = next((p for p in _CANDIDATE_DIRS if p.exists()), _CANDIDATE_DIRS[0])
print("Preprocessed data dir:", PREPROCESSED_DIR, "| exists:", PREPROCESSED_DIR.exists())

# Optional: directory of ORIGINAL fundus images. Only needed for the optional
# quantitative lesion-mask XAI section. Leave as None for the standard run.
RAW_IMAGES_DIR = None

# ----- Output locations -----
EXPERIMENT_NAME = "exp08_smote_served_npy"
RESULTS_DIR = Path("results") / EXPERIMENT_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FROZEN_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_frozen_best.keras"
MODEL_FINETUNE_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_finetune_best.keras"
HISTORY_FROZEN_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_history_frozen.json"
HISTORY_FINETUNE_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_history_finetune.json"
CONFIG_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_config.json"

TEST_METRICS_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_test_metrics.json"
TEST_REPORT_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_test_classification_report.csv"
TEST_CM_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_test_confusion_matrix.csv"
TEST_CM_NORMALIZED_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_test_confusion_matrix_normalized.csv"
TEST_CM_RAW_PNG_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_test_confusion_matrix_raw.png"
TEST_CM_NORMALIZED_PNG_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_test_confusion_matrix_normalized.png"
TEST_ROC_CURVE_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_test_roc_curves.png"

GRADCAM_DIR = RESULTS_DIR / "gradcam_heatmaps"
GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

# Lesion masks are optional and absent for APTOS; quantitative XAI is skipped without them.
LESION_MASK_DIR = RESULTS_DIR / "lesion_masks"
XAI_METRICS_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_gradcam_xai_metrics.csv"
XAI_SUMMARY_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_gradcam_xai_summary.json"

print("Results folder:", RESULTS_DIR)


Preprocessed data dir: data/preprocessed | exists: True
Results folder: results/exp08_smote_served_npy


## 2. Load Served Preprocessed Data

The six arrays were preprocessed and SMOTE-balanced upstream. We load them directly
and rescale pixels from `[0, 1]` to `[0, 255]` so the downstream augmentation and
EfficientNetB0 backbone behave exactly as in `exp08`.

In [ ]:
import numpy as np
from collections import Counter

# These arrays are the output of the dr_grading preprocessing pipeline:
#   resize -> LAB-CLAHE -> median denoise -> float32 [0, 1], with SMOTE applied
#   to the TRAINING split only. No raw images are read in this notebook.
X_train = np.load(PREPROCESSED_DIR / "X_train.npy").astype(np.float32)
X_val = np.load(PREPROCESSED_DIR / "X_val.npy").astype(np.float32)
X_test = np.load(PREPROCESSED_DIR / "X_test.npy").astype(np.float32)

y_train = np.load(PREPROCESSED_DIR / "y_train.npy").astype(np.int64)
y_val = np.load(PREPROCESSED_DIR / "y_val.npy").astype(np.int64)
y_test = np.load(PREPROCESSED_DIR / "y_test.npy").astype(np.int64)

print("Loaded served arrays:")
print("  X_train:", X_train.shape, X_train.dtype,
      "range", (float(X_train.min()), float(X_train.max())))
print("  X_val:  ", X_val.shape)
print("  X_test: ", X_test.shape)


def _to_255_range(arr):
    """Rescale [0, 1] pixels to [0, 255] float32 for EfficientNet / augmentation."""
    return np.clip(arr * 255.0, 0.0, 255.0).astype(np.float32)


# The pipeline stores pixels as float32 in [0, 1]. Keras EfficientNetB0 has its own
# internal rescaling/normalization that expects [0, 255], and the augmentation layers
# below use value_range=(0, 255). Rescale once so the rest of the notebook is identical
# to the in-memory exp08 run. Guarded so already-[0, 255] arrays pass through untouched.
if float(X_train.max()) <= 1.0 + 1e-6:
    print("\nDetected [0, 1] inputs -> rescaling to [0, 255].")
    X_train = _to_255_range(X_train)
    X_val = _to_255_range(X_val)
    X_test = _to_255_range(X_test)
else:
    print("\nInputs already exceed [0, 1]; assuming [0, 255], no rescaling applied.")

# SMOTE was applied upstream, so the training set is already balanced.
# These names match the variables the training cells expect.
X_train_fit = X_train
y_train_fit = y_train
TRAIN_CLASS_WEIGHT = None  # already balanced -> do not double-compensate with class weights.

# Original image ids are not stored in the served arrays.
test_id_codes = None

print("\nClass distributions:")
print("  train (post-SMOTE):", dict(sorted(Counter(y_train_fit).items())))
print("  val:               ", dict(sorted(Counter(y_val).items())))
print("  test:              ", dict(sorted(Counter(y_test).items())))
print("\nFinal X_train range:", (float(X_train_fit.min()), float(X_train_fit.max())),
      "| TRAIN_CLASS_WEIGHT:", TRAIN_CLASS_WEIGHT)


## 3. Augmentation

`value_range=(0, 255)` matches the rescaled pixel range from the load cell.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

augmentation_layers = [
    layers.RandomRotation(factor=15/360),
    layers.RandomFlip(mode="horizontal_and_vertical"),
    layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
    # value_range matches X_train/X_val range [0, 255]
    layers.RandomBrightness(factor=0.15, value_range=(0, 255)),
    layers.RandomContrast(factor=0.15),
]

# RandomErasing is not available in every TensorFlow/Keras version.
# Add it only when the local environment supports it.
if hasattr(layers, "RandomErasing"):
    augmentation_layers.append(
        layers.RandomErasing(
            factor=0.10,
            scale=(0.01, 0.04),
            fill_value=0.0,
            value_range=(0, 255)
        )
    )
else:
    print("RandomErasing is not available in this TensorFlow version; continuing without cutout.")

data_augmentation = tf.keras.Sequential(
    augmentation_layers,
    name="data_augmentation"
)


## 4. Validation QWK Callback

Quadratic weighted kappa is computed at each epoch end via a callback so `ModelCheckpoint` / `EarlyStopping` can monitor `val_qwk`.

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import cohen_kappa_score

class ValidationQWK(tf.keras.callbacks.Callback):
    """
    Computes validation quadratic weighted kappa after each epoch.

    Do not implement QWK as a normal Keras metric using .numpy() inside
    update_state(). Keras often trains in graph mode, where tensors are symbolic.
    This callback predicts on the validation set after each epoch, so it can use
    sklearn safely and can still expose logs['val_qwk'] for checkpointing and
    early stopping.
    """
    def __init__(self, validation_data, batch_size=32):
        super().__init__()
        self.X_val, self.y_val = validation_data
        self.batch_size = batch_size

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        y_pred_probs = self.model.predict(
            self.X_val,
            batch_size=self.batch_size,
            verbose=0
        )
        y_pred = np.argmax(y_pred_probs, axis=1)

        y_true = self.y_val
        if len(y_true.shape) > 1:
            y_true = np.argmax(y_true, axis=1)

        qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        logs["val_qwk"] = float(qwk)

        print(f" - val_qwk: {qwk:.4f}")

print("ValidationQWK callback ready.")


ValidationQWK callback ready.


## 5. Model Architecture

EfficientNetB0 backbone + Dense(256)→BN→Dropout(0.5)→Dense(128)→Dropout(0.4) head.

In [ ]:
from tensorflow.keras import layers, models

NUM_CLASSES = 5
INPUT_SHAPE = (224, 224, 3)


def build_model(trainable_from_layer=None):
    """
    Build EfficientNetB0 with an improved classification head.

    trainable_from_layer=None freezes the whole backbone.
    trainable_from_layer=int unfreezes backbone layers from that index onward.
    BatchNorm layers are always kept frozen for stability on this small dataset.
    """
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=INPUT_SHAPE
    )

    if trainable_from_layer is None:
        base_model.trainable = False
    else:
        base_model.trainable = True
        for layer in base_model.layers[:trainable_from_layer]:
            layer.trainable = False
        for layer in base_model.layers[trainable_from_layer:]:
            layer.trainable = True

    # Always keep BatchNorm frozen/in inference mode.
    for layer in base_model.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    inputs = layers.Input(shape=INPUT_SHAPE, name="input_image")
    x = data_augmentation(inputs)

    # EfficientNetB0 in Keras includes its own preprocessing/rescaling internally
    # in recent TF/Keras versions. We keep images in [0,255] to match augmentation.
    x = base_model(x, training=False)

    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(256, activation="relu", name="dense_256")(x)
    x = layers.BatchNormalization(name="bn_head")(x)
    x = layers.Dropout(0.5, name="dropout_1")(x)
    x = layers.Dense(128, activation="relu", name="dense_128")(x)
    x = layers.Dropout(0.4, name="dropout_2")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="output")(x)

    return models.Model(inputs, outputs, name="EfficientNetB0_Improved")


model = build_model(trainable_from_layer=None)
model.summary()


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "EfficientNetB0_Improved"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_256 (Dense)               │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_head (BatchNormalization)    │ (None, 256)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_128 (Dense)               │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,412,072 (16.83 MB)

 Trainable params: 361,989 (1.38 MB)

 Non-trainable params: 4,050,083 (15.45 MB)

## 5b. Loss Function — Sparse Categorical Focal Loss

Focal loss down-weights easy, already-correct samples and concentrates the gradient on the hard ones — here, the under-performing **Severe** class. It keeps the same 5-way softmax output, so evaluation, AUC, and Grad-CAM are unchanged.

`FOCAL_GAMMA` controls the focusing strength. `FOCAL_ALPHA` (per-class weighting) is left as `None`: the served data is already SMOTE-balanced, so adding per-class weights on top would double-compensate for class frequency. Gamma alone targets hard examples without re-weighting classes.

In [ ]:
import tensorflow as tf


class SparseCategoricalFocalLoss(tf.keras.losses.Loss):
    """Multiclass focal loss for integer labels and softmax-probability outputs.

    loss = alpha_c * (1 - p_t)^gamma * (-log p_t)

    where p_t is the predicted probability of the true class c.
    gamma=0 with alpha=None recovers ordinary cross-entropy.
    """

    def __init__(self, gamma=2.0, alpha=None, name="sparse_categorical_focal_loss"):
        super().__init__(name=name)
        self.gamma = float(gamma)
        self.alpha = None if alpha is None else tf.constant(alpha, dtype=tf.float32)

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        num_classes = tf.shape(y_pred)[-1]
        y_true_oh = tf.one_hot(y_true, depth=num_classes)

        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        p_t = tf.reduce_sum(y_true_oh * y_pred, axis=-1)

        cross_entropy = -tf.math.log(p_t)
        modulating = tf.pow(1.0 - p_t, self.gamma)
        loss = modulating * cross_entropy

        if self.alpha is not None:
            alpha_t = tf.reduce_sum(y_true_oh * self.alpha, axis=-1)
            loss = alpha_t * loss

        return tf.reduce_mean(loss)


FOCAL_GAMMA = 2.0
# Per-class weights [No DR, Mild, Moderate, Severe, Proliferative]. Kept None because
# the served training data is already SMOTE-balanced; per-class alpha on top of that
# would double-compensate for class frequency (risking minority over-prediction).
FOCAL_ALPHA = None

loss_fn = SparseCategoricalFocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)
print("Training loss:", loss_fn.name, "| gamma:", FOCAL_GAMMA, "| alpha:", FOCAL_ALPHA)


Training loss: sparse_categorical_focal_loss | gamma: 2.0 | alpha: None


## 6. Phase 1 — Frozen Backbone Training

Cosine-decay LR, frozen backbone, checkpoint + early stopping on `val_qwk`.

In [ ]:
import json
import math

BATCH_SIZE = 32
FROZEN_EPOCHS = 60
steps_per_epoch_frozen = math.ceil(len(X_train_fit) / BATCH_SIZE)

# CosineDecay alpha is a fraction of the initial LR.
# 1e-3 * 1e-3 = 1e-6 final LR.
lr_schedule_frozen = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=steps_per_epoch_frozen * FROZEN_EPOCHS,
    alpha=1e-3
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule_frozen),
    loss=loss_fn,
    metrics=["accuracy"]
)

callbacks_frozen = [
    # Must run before ModelCheckpoint/EarlyStopping so logs['val_qwk'] exists.
    ValidationQWK(validation_data=(X_val, y_val), batch_size=BATCH_SIZE),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_FROZEN_PATH),
        monitor="val_qwk",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_qwk",
        mode="max",
        patience=20,
        restore_best_weights=True,
        verbose=1
    )
]

print("Starting frozen backbone training...")

history_frozen = model.fit(
    X_train_fit,
    y_train_fit,
    validation_data=(X_val, y_val),
    epochs=FROZEN_EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=TRAIN_CLASS_WEIGHT,
    callbacks=callbacks_frozen
)

# Convert numpy scalars to normal Python values for JSON serialization.
history_frozen_dict = {
    k: [float(x) for x in v]
    for k, v in history_frozen.history.items()
}

with open(HISTORY_FROZEN_PATH, "w") as f:
    json.dump(history_frozen_dict, f, indent=4)

print("Phase 1 complete. Best model saved to:", MODEL_FROZEN_PATH)


Starting frozen backbone training...
Epoch 1/60
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.4105 - loss: 1.2061 - val_qwk: 0.8005

Epoch 1: val_qwk improved from None to 0.80054, saving model to results/exp08_smote_served_npy/exp08_smote_served_npy_frozen_best.keras

Epoch 1: finished saving model to results/exp08_smote_served_npy/exp08_smote_served_npy_frozen_best.keras
225/225 ━━━━━━━━━━━━━━━━━━━━ 34s 81ms/step - accuracy: 0.4643 - loss: 0.9316 - val_accuracy: 0.6421 - val_loss: 0.4372 - val_qwk: 0.8005
Epoch 2/60
223/225 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5377 - loss: 0.6436 - val_qwk: 0.7891

Epoch 2: val_qwk did not improve from 0.80054
225/225 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.5485 - loss: 0.6166 - val_accuracy: 0.6557 - val_loss: 0.4386 - val_qwk: 0.7891
Epoch 3/60
223/225 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.5751 - loss: 0.5457 - val_qwk: 0.8324

Epoch 3: val_qwk improved from 0.80054 to 0.83240, saving model to results/exp08_sm

## 7. Phase 2 — Fine-Tuning (Unfreeze from Layer 100)

Reloads the best frozen model, unfreezes the upper backbone, BatchNorm kept frozen.

In [ ]:
import math

FINETUNE_EPOCHS = 50
UNFREEZE_FROM_LAYER = 100

if not MODEL_FROZEN_PATH.exists():
    raise FileNotFoundError(f"Frozen model not found: {MODEL_FROZEN_PATH}")

# Load best frozen-phase model. No custom metric is needed because QWK is now a callback.
model = tf.keras.models.load_model(
    MODEL_FROZEN_PATH,
    compile=False
)

base_model = model.get_layer("efficientnetb0")
base_model.trainable = True

for layer in base_model.layers[:UNFREEZE_FROM_LAYER]:
    layer.trainable = False
for layer in base_model.layers[UNFREEZE_FROM_LAYER:]:
    layer.trainable = True

# Keep all BatchNorm layers frozen.
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(1 for layer in base_model.layers if layer.trainable)
print(f"Trainable EfficientNetB0 layers: {trainable_count} / {len(base_model.layers)}")

steps_per_epoch_ft = math.ceil(len(X_train_fit) / BATCH_SIZE)

lr_schedule_finetune = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=1e-4,
    first_decay_steps=steps_per_epoch_ft * 5,
    t_mul=2.0,
    m_mul=0.9,
    alpha=1e-3
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule_finetune),
    loss=loss_fn,
    metrics=["accuracy"]
)

callbacks_finetune = [
    ValidationQWK(validation_data=(X_val, y_val), batch_size=BATCH_SIZE),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_FINETUNE_PATH),
        monitor="val_qwk",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_qwk",
        mode="max",
        patience=15,
        restore_best_weights=True,
        verbose=1
    )
]

print("Starting fine-tuning...")

history_finetune = model.fit(
    X_train_fit,
    y_train_fit,
    validation_data=(X_val, y_val),
    epochs=FINETUNE_EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=TRAIN_CLASS_WEIGHT,
    callbacks=callbacks_finetune
)

history_finetune_dict = {
    k: [float(x) for x in v]
    for k, v in history_finetune.history.items()
}

with open(HISTORY_FINETUNE_PATH, "w") as f:
    json.dump(history_finetune_dict, f, indent=4)

print("Phase 2 complete. Best model saved to:", MODEL_FINETUNE_PATH)


## 8. Evaluation

Test-set accuracy, macro/weighted F1, QWK, multiclass AUC (OVR/OVO), confusion matrices, and ROC curves.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    auc
)
from sklearn.preprocessing import label_binarize

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
CLASS_IDS = np.arange(NUM_CLASSES)

# Image ids are not part of the served preprocessed arrays.
# `test_id_codes` is defined (as None) in the data-loading cell.

# Prefer the fine-tuned model. If fine-tuning was skipped or failed, evaluate frozen model.
if MODEL_FINETUNE_PATH.exists():
    EVAL_MODEL_PATH = MODEL_FINETUNE_PATH
    print("Evaluating fine-tuned model on TEST set:", EVAL_MODEL_PATH)
elif MODEL_FROZEN_PATH.exists():
    EVAL_MODEL_PATH = MODEL_FROZEN_PATH
    print("Fine-tuned model not found. Evaluating frozen model on TEST set:", EVAL_MODEL_PATH)
else:
    raise FileNotFoundError("No saved model found for evaluation.")

best_model = tf.keras.models.load_model(
    EVAL_MODEL_PATH,
    compile=False
)

# Test-time augmentation: average predictions over label-preserving views
# (flips / 180-degree rotation). A small, inference-only accuracy boost.
USE_TTA = True


def tta_predict(model, X, batch_size=BATCH_SIZE):
    views = [
        X,                                       # original
        np.ascontiguousarray(X[:, :, ::-1, :]),  # horizontal flip
        np.ascontiguousarray(X[:, ::-1, :, :]),  # vertical flip
        np.ascontiguousarray(X[:, ::-1, ::-1, :]),  # 180-degree rotation
    ]
    probs = np.zeros((len(X), NUM_CLASSES), dtype=np.float32)
    for view in views:
        probs += model.predict(view, batch_size=batch_size, verbose=0)
    return probs / len(views)


if USE_TTA:
    print("Predicting with test-time augmentation (4 views)...")
    y_test_pred_probs = tta_predict(best_model, X_test, batch_size=BATCH_SIZE)
else:
    y_test_pred_probs = best_model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)

y_test_pred = np.argmax(y_test_pred_probs, axis=1)

# ---------- AUC ----------
# For multiclass grading, report one-vs-rest AUC per class plus macro/weighted OVR.
# OVO is also reported as a robustness check when sklearn supports it for the current labels.
y_test_bin = label_binarize(y_test, classes=CLASS_IDS)

auc_per_class_ovr = {}
for class_id, class_name in zip(CLASS_IDS, CLASS_NAMES):
    positives = int(y_test_bin[:, class_id].sum())
    negatives = int(len(y_test_bin) - positives)

    if positives > 0 and negatives > 0:
        auc_per_class_ovr[class_name] = float(
            roc_auc_score(y_test_bin[:, class_id], y_test_pred_probs[:, class_id])
        )
    else:
        auc_per_class_ovr[class_name] = None

def safe_auc_score(*args, **kwargs):
    try:
        return float(roc_auc_score(*args, **kwargs))
    except Exception as exc:
        print("AUC skipped:", exc)
        return None

auc_ovr_macro = safe_auc_score(
    y_test,
    y_test_pred_probs,
    labels=CLASS_IDS,
    multi_class="ovr",
    average="macro"
)

auc_ovr_weighted = safe_auc_score(
    y_test,
    y_test_pred_probs,
    labels=CLASS_IDS,
    multi_class="ovr",
    average="weighted"
)

auc_ovo_macro = safe_auc_score(
    y_test,
    y_test_pred_probs,
    labels=CLASS_IDS,
    multi_class="ovo",
    average="macro"
)

auc_ovo_weighted = safe_auc_score(
    y_test,
    y_test_pred_probs,
    labels=CLASS_IDS,
    multi_class="ovo",
    average="weighted"
)

auc_ovr_micro = safe_auc_score(
    y_test_bin,
    y_test_pred_probs,
    average="micro"
)

test_metrics = {
    "evaluated_model": str(EVAL_MODEL_PATH),
    "evaluation_set": "test",
    "prediction_type": "softmax probabilities converted to class labels by argmax",
    "accuracy": float(accuracy_score(y_test, y_test_pred)),
    "macro_f1": float(f1_score(y_test, y_test_pred, average="macro", zero_division=0)),
    "macro_precision": float(precision_score(y_test, y_test_pred, average="macro", zero_division=0)),
    "macro_recall": float(recall_score(y_test, y_test_pred, average="macro", zero_division=0)),
    "weighted_f1": float(f1_score(y_test, y_test_pred, average="weighted", zero_division=0)),
    "quadratic_weighted_kappa": float(cohen_kappa_score(y_test, y_test_pred, weights="quadratic")),
    "auc_ovr_macro": auc_ovr_macro,
    "auc_ovr_weighted": auc_ovr_weighted,
    "auc_ovr_micro": auc_ovr_micro,
    "auc_ovo_macro": auc_ovo_macro,
    "auc_ovo_weighted": auc_ovo_weighted,
    "auc_per_class_ovr": auc_per_class_ovr
}

with open(TEST_METRICS_PATH, "w") as f:
    json.dump(test_metrics, f, indent=4)

print("=" * 40)
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")
print("=" * 40)

# ---------- Classification report ----------
test_report = classification_report(
    y_test,
    y_test_pred,
    labels=CLASS_IDS,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

test_report_df = pd.DataFrame(test_report).transpose()
test_report_df.to_csv(TEST_REPORT_PATH)

# ---------- Confusion matrix ----------
test_cm = confusion_matrix(y_test, y_test_pred, labels=CLASS_IDS)
test_cm_norm = confusion_matrix(y_test, y_test_pred, labels=CLASS_IDS, normalize="true")

test_cm_df = pd.DataFrame(
    test_cm,
    index=[f"true_{c}" for c in CLASS_NAMES],
    columns=[f"pred_{c}" for c in CLASS_NAMES]
)

test_cm_norm_df = pd.DataFrame(
    test_cm_norm,
    index=[f"true_{c}" for c in CLASS_NAMES],
    columns=[f"pred_{c}" for c in CLASS_NAMES]
)

test_cm_df.to_csv(TEST_CM_PATH)
test_cm_norm_df.to_csv(TEST_CM_NORMALIZED_PATH)

fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=test_cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, values_format="d", xticks_rotation=45, colorbar=False, cmap="Blues")
ax.set_title("Test Confusion Matrix — Counts")
plt.tight_layout()
plt.savefig(TEST_CM_RAW_PNG_PATH, dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=test_cm_norm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, values_format=".2f", xticks_rotation=45, colorbar=True, cmap="Blues")
ax.set_title("Test Confusion Matrix — Row Normalized")
plt.tight_layout()
plt.savefig(TEST_CM_NORMALIZED_PNG_PATH, dpi=200, bbox_inches="tight")
plt.show()

# ---------- ROC curves ----------
plt.figure(figsize=(8, 7))

for class_id, class_name in zip(CLASS_IDS, CLASS_NAMES):
    positives = int(y_test_bin[:, class_id].sum())
    negatives = int(len(y_test_bin) - positives)

    if positives == 0 or negatives == 0:
        continue

    fpr, tpr, _ = roc_curve(y_test_bin[:, class_id], y_test_pred_probs[:, class_id])
    roc_auc_value = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_name} AUC={roc_auc_value:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("One-vs-Rest ROC Curves on Test Set")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(TEST_ROC_CURVE_PATH, dpi=200, bbox_inches="tight")
plt.show()

print("Saved test metrics:", TEST_METRICS_PATH)
print("Saved test report:", TEST_REPORT_PATH)
print("Saved test confusion matrix:", TEST_CM_PATH)
print("Saved normalized test confusion matrix:", TEST_CM_NORMALIZED_PATH)
print("Saved raw confusion matrix plot:", TEST_CM_RAW_PNG_PATH)
print("Saved normalized confusion matrix plot:", TEST_CM_NORMALIZED_PNG_PATH)
print("Saved ROC curve plot:", TEST_ROC_CURVE_PATH)

print(test_report_df)
print(test_cm_df)
print(test_cm_norm_df)


# ---------- Per-class sensitivity & specificity + referable-DR readout ----------
# Derived one-vs-rest from the test confusion matrix (test_cm: rows=true, cols=pred).
per_class_sensitivity = {}
per_class_specificity = {}
cm_total = int(test_cm.sum())

print("\nPer-class one-vs-rest sensitivity & specificity:")
print(f"  {'Class':<18} {'Sensitivity':>11} {'Specificity':>12}")
for cid, cname in zip(CLASS_IDS, CLASS_NAMES):
    tp = int(test_cm[cid, cid])
    fn = int(test_cm[cid, :].sum()) - tp
    fp = int(test_cm[:, cid].sum()) - tp
    tn = cm_total - tp - fn - fp

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    per_class_sensitivity[cname] = float(sensitivity)
    per_class_specificity[cname] = float(specificity)
    print(f"  {cname:<18} {sensitivity:>11.3f} {specificity:>12.3f}")

# Referable DR = grade >= 2 (Moderate or worse): the core screening decision.
referable_true = (y_test >= 2).astype(int)
referable_pred = (y_test_pred >= 2).astype(int)
ref_sensitivity = recall_score(referable_true, referable_pred, pos_label=1, zero_division=0)
ref_specificity = recall_score(referable_true, referable_pred, pos_label=0, zero_division=0)
print(f"\nReferable DR (grade >= 2)  sensitivity: {ref_sensitivity:.3f} | specificity: {ref_specificity:.3f}")

test_metrics["per_class_sensitivity"] = per_class_sensitivity
test_metrics["per_class_specificity"] = per_class_specificity
test_metrics["referable_dr_sensitivity"] = float(ref_sensitivity)
test_metrics["referable_dr_specificity"] = float(ref_specificity)
test_metrics["test_time_augmentation"] = bool(USE_TTA)
with open(TEST_METRICS_PATH, "w") as f:
    json.dump(test_metrics, f, indent=4)


## 9. Grad-CAM Explainability (Qualitative)

Heatmaps from the last EfficientNetB0 conv layer for correct and misclassified test examples. Files are named by test index because the served arrays carry no image ids.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from tensorflow.keras import layers

GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

def find_last_conv_layer_name_for_gradcam(model, nested_model_name="efficientnetb0"):
    """Return the last 4D layer inside EfficientNetB0. Prefer top_conv if present."""
    base = model.get_layer(nested_model_name)

    try:
        base.get_layer("top_conv")
        return "top_conv"
    except ValueError:
        pass

    for layer in reversed(base.layers):
        try:
            if len(layer.output.shape) == 4:
                return layer.name
        except Exception:
            continue

    raise ValueError("Could not find a 4D convolutional layer for Grad-CAM.")


LAST_CONV_LAYER_NAME = find_last_conv_layer_name_for_gradcam(best_model)
print("Using Grad-CAM target layer:", LAST_CONV_LAYER_NAME)


def run_classification_head(model, base_outputs, training=False):
    """Run the part of the model after EfficientNetB0."""
    x = model.get_layer("gap")(base_outputs)
    x = model.get_layer("dense_256")(x)
    x = model.get_layer("bn_head")(x, training=training)
    x = model.get_layer("dropout_1")(x, training=training)
    x = model.get_layer("dense_128")(x)
    x = model.get_layer("dropout_2")(x, training=training)
    return model.get_layer("output")(x)


def make_gradcam_heatmap(
    model,
    image,
    class_index=None,
    last_conv_layer_name=LAST_CONV_LAYER_NAME
):
    """
    Generate a Grad-CAM heatmap for one image.

    image: one preprocessed image with shape (224, 224, 3), values in [0, 255].
    class_index:
      - None: explain the predicted class.
      - integer: explain a specific target class.
    """
    if image.ndim == 3:
        image_batch = np.expand_dims(image, axis=0)
    else:
        image_batch = image

    image_batch = tf.convert_to_tensor(image_batch, dtype=tf.float32)

    augmentation_layer = model.get_layer("data_augmentation")
    base_model = model.get_layer("efficientnetb0")
    target_layer = base_model.get_layer(last_conv_layer_name)

    conv_model = tf.keras.Model(
        inputs=base_model.input,
        outputs=[target_layer.output, base_model.output]
    )

    with tf.GradientTape() as tape:
        # data_augmentation is included in the trained model, but it is disabled with training=False.
        x = augmentation_layer(image_batch, training=False)
        conv_outputs, base_outputs = conv_model(x, training=False)
        tape.watch(conv_outputs)

        predictions = run_classification_head(model, base_outputs, training=False)

        if class_index is None:
            class_index = int(tf.argmax(predictions[0]).numpy())

        class_score = predictions[:, class_index]

    grads = tape.gradient(class_score, conv_outputs)

    if grads is None:
        raise RuntimeError("Gradients are None. Check that the target layer is connected to the classification output.")

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    max_value = tf.reduce_max(heatmap)

    if float(max_value.numpy()) > 0:
        heatmap = heatmap / max_value

    heatmap = heatmap.numpy().astype(np.float32)
    heatmap = cv2.resize(
        heatmap,
        (image_batch.shape[2], image_batch.shape[1]),
        interpolation=cv2.INTER_LINEAR
    )

    return heatmap, int(class_index), predictions.numpy()[0]


def overlay_gradcam(image, heatmap, alpha=0.35):
    """Overlay a normalized heatmap on a fundus image."""
    image_uint8 = np.clip(image, 0, 255).astype(np.uint8)
    heatmap_uint8 = np.uint8(255 * np.clip(heatmap, 0, 1))
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(image_uint8, 1 - alpha, heatmap_color, alpha, 0)
    return overlay


def select_gradcam_indices(y_true, y_pred, per_class=2, max_examples=15, random_state=42):
    """
    Pick a small but useful set:
    - up to per_class correct examples for each class
    - up to per_class mistakes for each true class
    """
    rng = np.random.default_rng(random_state)
    selected = []

    for class_id in CLASS_IDS:
        correct = np.where((y_true == class_id) & (y_pred == class_id))[0]
        wrong = np.where((y_true == class_id) & (y_pred != class_id))[0]

        if len(correct) > 0:
            selected.extend(rng.choice(correct, size=min(per_class, len(correct)), replace=False).tolist())

        if len(wrong) > 0:
            selected.extend(rng.choice(wrong, size=min(per_class, len(wrong)), replace=False).tolist())

    selected = list(dict.fromkeys(selected))
    return selected[:max_examples]


GRADCAM_CLASS_MODE = "predicted"  # "predicted" explains the model decision; "true" explains the ground-truth class.
GRADCAM_EXAMPLES_PER_CLASS = 2
MAX_GRADCAM_EXAMPLES = 15

gradcam_indices = select_gradcam_indices(
    y_test,
    y_test_pred,
    per_class=GRADCAM_EXAMPLES_PER_CLASS,
    max_examples=MAX_GRADCAM_EXAMPLES
)

if len(gradcam_indices) == 0:
    print("No Grad-CAM examples selected.")
else:
    gradcam_records = []
    n = len(gradcam_indices)

    fig, axes = plt.subplots(n, 3, figsize=(12, 3.5 * n))

    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, local_idx in enumerate(gradcam_indices):
        image = X_test[local_idx]
        true_class = int(y_test[local_idx])
        pred_class = int(y_test_pred[local_idx])

        if GRADCAM_CLASS_MODE == "true":
            target_class = true_class
        else:
            target_class = pred_class

        heatmap, explained_class, probs = make_gradcam_heatmap(
            best_model,
            image,
            class_index=target_class
        )

        overlay = overlay_gradcam(image, heatmap)
        image_id = (
            f"test_{local_idx:05d}"
            if test_id_codes is None
            else str(test_id_codes[local_idx])
        )

        overlay_path = GRADCAM_DIR / (
            f"{image_id}_true-{true_class}_pred-{pred_class}_explained-{explained_class}_overlay.png"
        )
        heatmap_path = GRADCAM_DIR / (
            f"{image_id}_true-{true_class}_pred-{pred_class}_explained-{explained_class}_heatmap.png"
        )

        plt.imsave(overlay_path, overlay)
        plt.imsave(heatmap_path, heatmap, cmap="jet")

        axes[row, 0].imshow(np.clip(image / 255.0, 0.0, 1.0))
        axes[row, 0].set_title(f"{image_id}\ntrue={CLASS_NAMES[true_class]}")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(heatmap, cmap="jet")
        axes[row, 1].set_title(f"Grad-CAM\nexplains={CLASS_NAMES[explained_class]}")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title(f"pred={CLASS_NAMES[pred_class]}\nconf={probs[pred_class]:.3f}")
        axes[row, 2].axis("off")

        gradcam_records.append({
            "test_local_index": int(local_idx),
            "id_code": image_id,
            "true_class_id": true_class,
            "true_class_name": CLASS_NAMES[true_class],
            "pred_class_id": pred_class,
            "pred_class_name": CLASS_NAMES[pred_class],
            "explained_class_id": explained_class,
            "explained_class_name": CLASS_NAMES[explained_class],
            "predicted_confidence": float(probs[pred_class]),
            "overlay_path": str(overlay_path),
            "heatmap_path": str(heatmap_path)
        })

    plt.tight_layout()

    gradcam_grid_path = GRADCAM_DIR / f"{EXPERIMENT_NAME}_gradcam_examples_grid.png"
    plt.savefig(gradcam_grid_path, dpi=200, bbox_inches="tight")
    plt.show()

    gradcam_index_path = GRADCAM_DIR / f"{EXPERIMENT_NAME}_gradcam_examples_index.csv"
    pd.DataFrame(gradcam_records).to_csv(gradcam_index_path, index=False)

    print("Saved Grad-CAM example grid:", gradcam_grid_path)
    print("Saved Grad-CAM example index:", gradcam_index_path)
    print("Saved individual Grad-CAM overlays to:", GRADCAM_DIR)

## 10. Optional Quantitative XAI (Pointing Game / IoU)

Requires pixel-level lesion masks **and** original image ids — neither is available from the served arrays for APTOS, so this section self-skips. Set `RAW_IMAGES_DIR`, populate `LESION_MASK_DIR`, and provide `test_id_codes` (e.g. from IDRiD) to enable it.

In [ ]:
import json
from tqdm.auto import tqdm

# Quantitative localization evaluation.
# This requires lesion masks. APTOS 2019 does not ship with lesion masks.
# Put masks in LESION_MASK_DIR, preferably with filenames containing the same id_code.
# The loader recursively combines all matching masks for an image into one binary lesion mask.

LESION_MASK_EXTENSIONS = [".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".npy"]
HEATMAP_TOP_PERCENT = 15.0  # threshold top 15% of heatmap pixels for IoU-style evaluation
MAX_XAI_EVAL_IMAGES = None  # set to an integer for a faster subset, e.g. 100


def get_crop_bbox_from_rgb(img_rgb, threshold=10):
    """Return crop box using the same black-border logic as preprocessing."""
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    valid = gray > threshold

    if not valid.any():
        h, w = gray.shape
        return 0, h, 0, w

    rows = np.where(valid.any(axis=1))[0]
    cols = np.where(valid.any(axis=0))[0]
    return int(rows[0]), int(rows[-1] + 1), int(cols[0]), int(cols[-1] + 1)


def resize_binary_mask_with_padding(mask, target_size=224):
    """Resize a binary mask with the same aspect-ratio padding used for images."""
    mask = (mask > 0).astype(np.uint8)

    h, w = mask.shape[:2]
    scale = target_size / max(h, w)

    new_w = int(round(w * scale))
    new_h = int(round(h * scale))

    resized = cv2.resize(
        mask,
        (new_w, new_h),
        interpolation=cv2.INTER_NEAREST
    )

    padded = np.zeros((target_size, target_size), dtype=np.uint8)

    y_off = (target_size - new_h) // 2
    x_off = (target_size - new_w) // 2

    padded[y_off:y_off + new_h, x_off:x_off + new_w] = resized

    return padded.astype(bool)


def read_mask_file(mask_path):
    """Read one mask file as a binary numpy array."""
    mask_path = Path(mask_path)

    if mask_path.suffix.lower() == ".npy":
        mask = np.load(mask_path)
    else:
        mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

    if mask is None:
        return None

    if mask.ndim == 3:
        # If the mask is RGB/RGBA, combine non-zero channels.
        mask = np.max(mask, axis=-1)

    return (mask > 0).astype(np.uint8)


def preprocess_mask_to_model_space(mask, original_image_path, img_size=224):
    """
    Align lesion masks to the model input space.

    If mask has original image size, apply the same crop and padding as the fundus image.
    If mask already has model size, keep it.
    Otherwise, resize with nearest-neighbor as a fallback.
    """
    if mask.shape[:2] == (img_size, img_size):
        return (mask > 0).astype(bool)

    original_bgr = cv2.imread(str(original_image_path))

    if original_bgr is not None:
        original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)

        if mask.shape[:2] == original_rgb.shape[:2]:
            y0, y1, x0, x1 = get_crop_bbox_from_rgb(original_rgb, threshold=10)
            mask = mask[y0:y1, x0:x1]
            return resize_binary_mask_with_padding(mask, target_size=img_size)

    # Fallback for already-cropped masks or unknown layouts.
    resized = cv2.resize(
        (mask > 0).astype(np.uint8),
        (img_size, img_size),
        interpolation=cv2.INTER_NEAREST
    )

    return resized.astype(bool)


def find_mask_files_for_id(id_code, mask_root):
    """Find all lesion mask files whose filename contains the test image id_code."""
    mask_root = Path(mask_root)

    if not mask_root.exists():
        return []

    id_code = str(id_code)
    matches = []

    for ext in LESION_MASK_EXTENSIONS:
        matches.extend(mask_root.rglob(f"{id_code}{ext}"))
        matches.extend(mask_root.rglob(f"{id_code}_*{ext}"))
        matches.extend(mask_root.rglob(f"*{id_code}*{ext}"))

    # De-duplicate while preserving order.
    return list(dict.fromkeys(matches))


def load_combined_lesion_mask(id_code, original_image_path, mask_root, img_size=224):
    """Load and OR-combine all matching lesion masks for one image."""
    mask_files = find_mask_files_for_id(id_code, mask_root)

    if len(mask_files) == 0:
        return None, []

    combined = np.zeros((img_size, img_size), dtype=bool)
    loaded_files = []

    for mask_file in mask_files:
        raw_mask = read_mask_file(mask_file)

        if raw_mask is None:
            continue

        processed_mask = preprocess_mask_to_model_space(
            raw_mask,
            original_image_path=original_image_path,
            img_size=img_size
        )

        combined |= processed_mask
        loaded_files.append(str(mask_file))

    if len(loaded_files) == 0 or combined.sum() == 0:
        return None, loaded_files

    return combined, loaded_files


def compute_pointing_game_and_iou(heatmap, lesion_mask, top_percent=15.0):
    """
    Compute paper-style localization metrics:
    - Pointing Game hit: argmax heatmap pixel is inside the lesion mask.
    - Energy inside mask: percentage of heatmap mass inside the lesion mask.
    - IoU/Dice/precision/recall after thresholding top heatmap pixels.
    """
    heatmap = np.asarray(heatmap, dtype=np.float32)
    lesion_mask = np.asarray(lesion_mask).astype(bool)

    if lesion_mask.sum() == 0:
        return None

    peak_y, peak_x = np.unravel_index(np.argmax(heatmap), heatmap.shape)
    pointing_hit = int(lesion_mask[peak_y, peak_x])

    heatmap_sum = float(heatmap.sum())
    if heatmap_sum > 0:
        energy_inside = float(heatmap[lesion_mask].sum() / heatmap_sum)
    else:
        energy_inside = 0.0

    if float(heatmap.max()) <= 0:
        heatmap_binary = np.zeros_like(lesion_mask, dtype=bool)
    else:
        threshold = np.percentile(heatmap, 100.0 - top_percent)
        heatmap_binary = heatmap >= threshold

    intersection = np.logical_and(heatmap_binary, lesion_mask).sum()
    union = np.logical_or(heatmap_binary, lesion_mask).sum()

    heatmap_area = heatmap_binary.sum()
    lesion_area = lesion_mask.sum()

    iou = float(intersection / union) if union > 0 else 0.0
    precision = float(intersection / heatmap_area) if heatmap_area > 0 else 0.0
    recall = float(intersection / lesion_area) if lesion_area > 0 else 0.0

    if precision + recall > 0:
        dice = float(2 * precision * recall / (precision + recall))
    else:
        dice = 0.0

    return {
        "pointing_hit": pointing_hit,
        "peak_y": int(peak_y),
        "peak_x": int(peak_x),
        "energy_inside_mask": energy_inside,
        "iou_top_percent": iou,
        "dice_top_percent": dice,
        "precision_top_percent": precision,
        "recall_top_percent": recall,
        "heatmap_top_percent": float(top_percent),
        "lesion_area_pixels": int(lesion_area),
        "heatmap_area_pixels": int(heatmap_area),
        "intersection_pixels": int(intersection),
        "union_pixels": int(union)
    }


_xai_ready = (
    LESION_MASK_DIR.exists()
    and test_id_codes is not None
    and RAW_IMAGES_DIR is not None
    and Path(RAW_IMAGES_DIR).exists()
)

if not _xai_ready:
    print("Skipping quantitative XAI evaluation (Pointing Game / IoU).")
    print("It needs lesion masks AND original image ids + images, which are not")
    print("part of the served preprocessed arrays.")
    print("LESION_MASK_DIR exists:", LESION_MASK_DIR.exists(),
          "| test_id_codes available:", test_id_codes is not None,
          "| RAW_IMAGES_DIR set:", RAW_IMAGES_DIR is not None)
    print("APTOS 2019 has severity labels but no lesion masks. Add IDRiD-style")
    print("pixel-annotated masks (and RAW_IMAGES_DIR) to run Pointing Game and IoU.")
else:
    xai_rows = []
    missing_masks = 0

    eval_indices = np.arange(len(X_test))

    if MAX_XAI_EVAL_IMAGES is not None:
        eval_indices = eval_indices[:MAX_XAI_EVAL_IMAGES]

    for local_idx in tqdm(eval_indices, desc="Grad-CAM XAI mask evaluation"):
        image_id = str(test_id_codes[local_idx])
        original_image_path = Path(RAW_IMAGES_DIR) / f"{image_id}.png"

        lesion_mask, mask_files = load_combined_lesion_mask(
            image_id,
            original_image_path=original_image_path,
            mask_root=LESION_MASK_DIR,
            img_size=IMG_SIZE
        )

        if lesion_mask is None:
            missing_masks += 1
            continue

        image = X_test[local_idx]
        true_class = int(y_test[local_idx])
        pred_class = int(y_test_pred[local_idx])

        # For explainability evaluation, explain the model's predicted class.
        heatmap, explained_class, probs = make_gradcam_heatmap(
            best_model,
            image,
            class_index=pred_class
        )

        metrics = compute_pointing_game_and_iou(
            heatmap,
            lesion_mask,
            top_percent=HEATMAP_TOP_PERCENT
        )

        if metrics is None:
            continue

        row = {
            "test_local_index": int(local_idx),
            "id_code": image_id,
            "true_class_id": true_class,
            "true_class_name": CLASS_NAMES[true_class],
            "pred_class_id": pred_class,
            "pred_class_name": CLASS_NAMES[pred_class],
            "explained_class_id": int(explained_class),
            "explained_class_name": CLASS_NAMES[int(explained_class)],
            "predicted_confidence": float(probs[pred_class]),
            "mask_file_count": int(len(mask_files)),
            "mask_files": "|".join(mask_files)
        }

        row.update(metrics)
        xai_rows.append(row)

    if len(xai_rows) == 0:
        print("No images with usable lesion masks were found.")
        print("Missing masks for images checked:", missing_masks)
    else:
        xai_df = pd.DataFrame(xai_rows)
        xai_df.to_csv(XAI_METRICS_PATH, index=False)

        xai_summary = {
            "evaluated_model": str(EVAL_MODEL_PATH),
            "lesion_mask_dir": str(LESION_MASK_DIR),
            "n_test_images_checked": int(len(eval_indices)),
            "n_images_with_usable_masks": int(len(xai_df)),
            "n_missing_or_empty_masks": int(missing_masks),
            "pointing_game_accuracy": float(xai_df["pointing_hit"].mean()),
            "mean_energy_inside_mask": float(xai_df["energy_inside_mask"].mean()),
            f"mean_iou_top_{HEATMAP_TOP_PERCENT:g}_percent": float(xai_df["iou_top_percent"].mean()),
            f"mean_dice_top_{HEATMAP_TOP_PERCENT:g}_percent": float(xai_df["dice_top_percent"].mean()),
            f"mean_precision_top_{HEATMAP_TOP_PERCENT:g}_percent": float(xai_df["precision_top_percent"].mean()),
            f"mean_recall_top_{HEATMAP_TOP_PERCENT:g}_percent": float(xai_df["recall_top_percent"].mean())
        }

        with open(XAI_SUMMARY_PATH, "w") as f:
            json.dump(xai_summary, f, indent=4)

        print("Saved Grad-CAM XAI metrics:", XAI_METRICS_PATH)
        print("Saved Grad-CAM XAI summary:", XAI_SUMMARY_PATH)
        print(json.dumps(xai_summary, indent=4))

        display(xai_df.head())

## 11. Training Curves

In [ ]:
import json
import matplotlib.pyplot as plt


def plot_history(history, title):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(title)

    if "accuracy" in history:
        axes[0].plot(history["accuracy"], label="train")
    if "val_accuracy" in history:
        axes[0].plot(history["val_accuracy"], label="val")
    axes[0].set_title("Accuracy")
    axes[0].legend()

    if "loss" in history:
        axes[1].plot(history["loss"], label="train")
    if "val_loss" in history:
        axes[1].plot(history["val_loss"], label="val")
    axes[1].set_title("Loss")
    axes[1].legend()

    # QWK is produced by the callback as validation-only metric.
    if "val_qwk" in history:
        axes[2].plot(history["val_qwk"], label="val_qwk")
    axes[2].set_title("Quadratic Weighted Kappa")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

if HISTORY_FROZEN_PATH.exists():
    with open(HISTORY_FROZEN_PATH) as f:
        h_frozen = json.load(f)
    plot_history(h_frozen, "Phase 1 — Frozen Backbone")
else:
    print("Frozen history not found:", HISTORY_FROZEN_PATH)

if HISTORY_FINETUNE_PATH.exists():
    with open(HISTORY_FINETUNE_PATH) as f:
        h_finetune = json.load(f)
    plot_history(h_finetune, "Phase 2 — Fine-Tuning")
else:
    print("Fine-tune history not found:", HISTORY_FINETUNE_PATH)


## 12. Experiment Config (Documentation)

In [ ]:
import json
from collections import Counter

# Provenance from the served arrays' manifest, when present.
manifest_path = PREPROCESSED_DIR / "preprocessing_manifest.json"
if manifest_path.exists():
    with open(manifest_path) as f:
        preprocessing_manifest = json.load(f)
else:
    preprocessing_manifest = None
    print("No preprocessing_manifest.json found next to the served arrays.")

train_distribution = {str(k): int(v) for k, v in sorted(Counter(y_train_fit).items())}
val_distribution = {str(k): int(v) for k, v in sorted(Counter(y_val).items())}
test_distribution = {str(k): int(v) for k, v in sorted(Counter(y_test).items())}

experiment_config = {
    "experiment_name": EXPERIMENT_NAME,
    "goal": (
        "Train and evaluate EfficientNetB0 for APTOS 2019 DR severity grading using "
        "preprocessed + SMOTE-balanced arrays served as .npy files, with paper-aligned "
        "evaluation, multiclass AUC, and Grad-CAM XAI."
    ),
    "data_source": {
        "preprocessed_dir": str(PREPROCESSED_DIR),
        "files": ["X_train.npy", "X_val.npy", "X_test.npy",
                  "y_train.npy", "y_val.npy", "y_test.npy"],
        "note": (
            "Preprocessing (resize -> LAB-CLAHE -> median denoise -> float32 [0,1]) and "
            "SMOTE (training split only) were applied upstream by the dr_grading pipeline. "
            "This notebook consumes the served arrays directly and rescales pixels to "
            "[0,255] for EfficientNet / augmentation."
        ),
    },
    "preprocessing_manifest": preprocessing_manifest,
    "dataset": "APTOS 2019",
    "image_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "num_classes": NUM_CLASSES,
    "loss": {
        "type": "sparse_categorical_focal_loss",
        "gamma": FOCAL_GAMMA,
        "alpha": FOCAL_ALPHA,
    },
    "test_time_augmentation": bool(USE_TTA),
    "reproducibility_seed": SEED,
    "train_distribution_post_smote": train_distribution,
    "val_distribution": val_distribution,
    "test_distribution": test_distribution,
    "train_class_weight": TRAIN_CLASS_WEIGHT,
    "unfreeze_from_layer": UNFREEZE_FROM_LAYER,
    "frozen_epochs": FROZEN_EPOCHS,
    "finetune_epochs": FINETUNE_EPOCHS,
    "evaluation_outputs": {
        "test_metrics_json": str(TEST_METRICS_PATH),
        "classification_report_csv": str(TEST_REPORT_PATH),
        "confusion_matrix_csv": str(TEST_CM_PATH),
        "confusion_matrix_normalized_csv": str(TEST_CM_NORMALIZED_PATH),
        "confusion_matrix_raw_png": str(TEST_CM_RAW_PNG_PATH),
        "confusion_matrix_normalized_png": str(TEST_CM_NORMALIZED_PNG_PATH),
        "roc_curve_png": str(TEST_ROC_CURVE_PATH),
        "gradcam_dir": str(GRADCAM_DIR),
        "xai_metrics_csv": str(XAI_METRICS_PATH),
        "xai_summary_json": str(XAI_SUMMARY_PATH),
        "lesion_mask_dir": str(LESION_MASK_DIR),
    },
    "paper_basis": [
        {
            "name": "Selvaraju et al. (2017), Grad-CAM: Visual Explanations from Deep Networks via Gradient-Based Localization.",
            "used_for": "Grad-CAM heatmap generation from the final convolutional layer.",
        },
        {
            "name": "Zhang et al. (2016), Top-down Neural Attention by Excitation Backprop.",
            "used_for": "Pointing Game localization metric.",
        },
        {
            "name": "Porwal et al. (2018), Indian Diabetic Retinopathy Image Dataset (IDRiD).",
            "used_for": "Lesion-mask based evaluation: microaneurysms, hemorrhages, hard exudates, and soft exudates.",
        },
        {
            "name": "Dharrao et al. (2025), AI-driven detection and classification of diabetic retinopathy stages using EfficientNetB0.",
            "used_for": "EfficientNetB0 DR severity-classification context.",
        },
    ],
}

with open(CONFIG_PATH, "w") as f:
    json.dump(experiment_config, f, indent=4)

print("Config saved to:", CONFIG_PATH)
print(json.dumps({k: v for k, v in experiment_config.items()
                  if k != "preprocessing_manifest"}, indent=4))


## 13. IDRiD Lesion-Mask IoU (Grad-CAM Localization vs. Ground Truth)

Quantitative explainability check for this model: do the Grad-CAM heatmaps actually
land on *real* diabetic-retinopathy lesions?

APTOS (used for train / val / test above) ships only image-level severity labels —
no pixel masks — so this uses **IDRiD**, the public dataset with pixel-level lesion
annotations, as an *external* localization benchmark:

- 81 fundus images (`.jpg`) with binary lesion masks (`.tif`) for **MA**
  (microaneurysms), **HE** (haemorrhages), **EX** (hard exudates), **SE** (soft
  exudates). The optic disc (`OD`) is normal anatomy, not a lesion, so it is excluded.
- Each IDRiD image is pushed through the **exact same preprocessing as APTOS**
  (resize→224 `INTER_AREA` → LAB-CLAHE → 3×3 median, kept in **BGR**, range
  `[0, 255]`) so the trained model sees inputs from its training distribution.
- The available lesion masks are OR-combined into one binary lesion map and resized
  to 224×224 with **nearest-neighbour** — the *same* plain resize the images get
  (the pipeline does no crop/pad), so heatmap and mask share pixel coordinates.
- For each image we Grad-CAM the model's **predicted** class (decision faithfulness),
  threshold the top `HEATMAP_TOP_PERCENT` of heatmap energy, and compute
  **IoU / Dice / precision / recall** against the lesion map, plus the **Pointing
  Game** hit rate (does the heatmap peak fall on a lesion?) and **energy-inside-mask**.
  IoU is also reported across a sweep of thresholds, since it is threshold-sensitive.

Outputs are tagged with `IDRID_MODEL_TAG` so each model's IDRiD results sit side by
side (the plain-SMOTE and SMOTE+focal notebooks share `RESULTS_DIR`). The data is
fetched automatically via `kagglehub` if not already present; otherwise set
`IDRID_ROOT` (next cell) to a local / Drive copy. The section self-skips cleanly if
neither is available, and reuses `make_gradcam_heatmap` (Section 9) and
`compute_pointing_game_and_iou` (Section 10) unchanged.


In [ ]:
# === IDRiD acquisition & discovery =========================================
# Locate an IDRiD "Segmentation" copy, or download it via kagglehub.
# Robust to mirror layout: original images are matched by *.jpg named IDRiD_<n>,
# masks by *.tif named IDRiD_<n>_<CODE> (MA / HE / EX / SE / OD), wherever they
# sit in the extracted tree. Images vs. masks are told apart purely by extension.
import os
import re
from pathlib import Path

# Optional explicit override. Point this at a folder that contains the IDRiD
# segmentation tree (original .jpg images + .tif lesion masks anywhere inside).
IDRID_ROOT = None  # e.g. Path("/content/drive/MyDrive/idrid") or Path("data/idrid")

# Public Kaggle mirror that includes the segmentation ground truths.
IDRID_KAGGLE_DATASET = "gyanpr02/indian-diabetic-retinopathy-image-datasetidrid"

_IDRID_CANDIDATE_ROOTS = [
    p for p in [
        IDRID_ROOT,
        Path("data/idrid"),
        Path("capstone-project/data/idrid"),
        Path("/content/drive/MyDrive/dr_severity_aptos/data/idrid"),
        Path("/content/idrid"),
    ] if p is not None
]

_IMG_ID_RE = re.compile(r"^(IDRiD_\d+)$", re.IGNORECASE)
_MASK_ID_RE = re.compile(r"^(IDRiD_\d+)_([A-Za-z]{2})$", re.IGNORECASE)


def discover_idrid_segmentation(root):
    """Return ({id: image_path}, {id: {CODE: mask_path}}) discovered under ``root``.

    Distinguishes images from masks purely by extension (.jpg vs .tif), so it
    works regardless of how a given mirror names its sub-folders. Ids are keyed
    lower-case so image and mask lookups always line up.
    """
    root = Path(root)
    images, masks = {}, {}
    if not root.exists():
        return images, masks

    for path in root.rglob("*"):
        if not path.is_file():
            continue
        suffix = path.suffix.lower()
        if suffix in (".jpg", ".jpeg"):
            match = _IMG_ID_RE.match(path.stem)
            if match:
                images.setdefault(match.group(1).lower(), path)
        elif suffix in (".tif", ".tiff"):
            match = _MASK_ID_RE.match(path.stem)
            if match:
                masks.setdefault(match.group(1).lower(), {})[match.group(2).upper()] = path
    return images, masks


def ensure_idrid_segmentation():
    """Find a local IDRiD copy, else download via kagglehub. -> (root, images, masks)."""
    for root in _IDRID_CANDIDATE_ROOTS:
        images, masks = discover_idrid_segmentation(root)
        if images and masks:
            print(f"Found IDRiD segmentation under: {root}")
            return root, images, masks

    print("No local IDRiD copy found. Attempting kagglehub download:", IDRID_KAGGLE_DATASET)
    try:
        try:
            import kagglehub
        except ImportError:
            import subprocess
            import sys
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
            import kagglehub
        download_path = Path(kagglehub.dataset_download(IDRID_KAGGLE_DATASET))
        images, masks = discover_idrid_segmentation(download_path)
        if images and masks:
            print(f"Downloaded IDRiD to: {download_path}")
            return download_path, images, masks
        print("Downloaded dataset but could not locate the segmentation tree in:", download_path)
    except Exception as exc:  # noqa: BLE001 - network / credentials / availability
        print("kagglehub download failed:", repr(exc))

    return None, {}, {}


idrid_root, idrid_images, idrid_masks = ensure_idrid_segmentation()

# An image is usable only if it carries at least one pathological-lesion mask.
LESION_CODES_FOR_IOU = ["MA", "HE", "EX", "SE"]  # exclude OD (optic disc = anatomy)
idrid_eval_ids = sorted(
    cid for cid in idrid_images
    if any(code in idrid_masks.get(cid, {}) for code in LESION_CODES_FOR_IOU)
)

IDRID_READY = len(idrid_eval_ids) > 0
print(
    f"IDRiD images discovered: {len(idrid_images)} | "
    f"with lesion masks: {len(idrid_eval_ids)} | ready: {IDRID_READY}"
)
if not IDRID_READY:
    print("\nTo enable this section, either:")
    print("  - provide Kaggle credentials so kagglehub can download, or")
    print("  - set IDRID_ROOT above to a folder containing the IDRiD segmentation subset")
    print("    (original .jpg images + .tif lesion masks). Source / manual download:")
    print("    https://ieee-dataport.org/open-access/indian-diabetic-retinopathy-image-dataset-idrid")


In [ ]:
# === IDRiD Grad-CAM IoU evaluation =========================================
# Reuses, unchanged: make_gradcam_heatmap (Section 9) and
# compute_pointing_game_and_iou (Section 10).
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Tag keeps this model's IDRiD outputs separate from the sibling notebook, which
# shares EXPERIMENT_NAME / RESULTS_DIR ("smote" vs "smote_focal").
IDRID_MODEL_TAG = "smote_focal"
IDRID_TOP_PERCENT = globals().get("HEATMAP_TOP_PERCENT", 15.0)  # primary threshold
IDRID_THRESHOLD_SWEEP = [5, 10, 15, 20, 25, 30]                 # IoU vs. threshold sweep
MAX_IDRID_IMAGES = None                                         # set an int for a quick subset
IDRID_GRID_EXAMPLES = 6                                         # qualitative panels to render

IDRID_XAI_METRICS_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_{IDRID_MODEL_TAG}_idrid_gradcam_iou_metrics.csv"
IDRID_XAI_SUMMARY_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_{IDRID_MODEL_TAG}_idrid_gradcam_iou_summary.json"
IDRID_GRID_PATH = GRADCAM_DIR / f"{EXPERIMENT_NAME}_{IDRID_MODEL_TAG}_idrid_gradcam_iou_grid.png"


def preprocess_idrid_image(image_path, image_size=IMG_SIZE):
    """Replicate the APTOS training preprocessing exactly (BGR, range [0, 255])."""
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        return None
    resized = cv2.resize(bgr, (image_size, image_size), interpolation=cv2.INTER_AREA)
    lab = cv2.cvtColor(resized, cv2.COLOR_BGR2LAB)
    lightness, a_channel, b_channel = cv2.split(lab)
    lightness = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lightness)
    enhanced = cv2.cvtColor(cv2.merge([lightness, a_channel, b_channel]), cv2.COLOR_LAB2BGR)
    denoised = cv2.medianBlur(enhanced, 3)
    # Pipeline yields float32 [0, 1]; the notebook then rescales to [0, 255] for the model.
    return np.clip(denoised.astype(np.float32), 0.0, 255.0)


def load_idrid_lesion_mask(mask_paths_by_code, codes=LESION_CODES_FOR_IOU, image_size=IMG_SIZE):
    """OR-combine the available lesion masks and align them to the 224x224 model space.

    Images are preprocessed with a plain resize to (224, 224), so masks get the same
    plain nearest-neighbour resize -> heatmap and mask share pixel coordinates.
    """
    combined = None
    used = []
    for code in codes:
        path = mask_paths_by_code.get(code)
        if path is None:
            continue
        raw = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        if raw is None:
            continue
        if raw.ndim == 3:
            raw = raw.max(axis=-1)
        binary = (raw > 0).astype(np.uint8)
        binary = cv2.resize(binary, (image_size, image_size), interpolation=cv2.INTER_NEAREST)
        combined = binary.astype(bool) if combined is None else (combined | binary.astype(bool))
        used.append(code)
    if combined is None or combined.sum() == 0:
        return None, used
    return combined, used


def _iou_at(heatmap, mask, top_percent):
    res = compute_pointing_game_and_iou(heatmap, mask, top_percent=top_percent)
    return float(res["iou_top_percent"]) if res else 0.0


if not IDRID_READY:
    print("Skipping IDRiD lesion-mask IoU evaluation (no IDRiD segmentation data available).")
else:
    eval_ids = idrid_eval_ids if MAX_IDRID_IMAGES is None else idrid_eval_ids[:MAX_IDRID_IMAGES]

    idrid_rows = []
    panels = []  # (image_rgb, heatmap, mask, id, pred_name, iou) for the qualitative grid

    for cid in tqdm(eval_ids, desc="IDRiD Grad-CAM IoU"):
        image = preprocess_idrid_image(idrid_images[cid])
        if image is None:
            continue

        lesion_mask, used_codes = load_idrid_lesion_mask(idrid_masks.get(cid, {}))
        if lesion_mask is None:
            continue

        # Explain the model's predicted class -> faithfulness of the decision.
        heatmap, explained_class, probs = make_gradcam_heatmap(best_model, image, class_index=None)

        metrics = compute_pointing_game_and_iou(heatmap, lesion_mask, top_percent=IDRID_TOP_PERCENT)
        if metrics is None:
            continue

        sweep = {
            f"iou_top_{t:g}pct": _iou_at(heatmap, lesion_mask, t)
            for t in IDRID_THRESHOLD_SWEEP
        }

        display_id = idrid_images[cid].stem
        row = {
            "id_code": display_id,
            "pred_class_id": int(explained_class),
            "pred_class_name": CLASS_NAMES[int(explained_class)],
            "pred_confidence": float(probs[int(explained_class)]),
            "lesion_codes": "+".join(used_codes),
        }
        row.update(metrics)
        row.update(sweep)
        idrid_rows.append(row)

        if len(panels) < IDRID_GRID_EXAMPLES:
            image_rgb = cv2.cvtColor(np.clip(image, 0, 255).astype(np.uint8), cv2.COLOR_BGR2RGB)
            panels.append(
                (image_rgb, heatmap, lesion_mask, display_id,
                 CLASS_NAMES[int(explained_class)], metrics["iou_top_percent"])
            )

    if len(idrid_rows) == 0:
        print("No IDRiD images produced usable masks/heatmaps.")
    else:
        idrid_df = pd.DataFrame(idrid_rows)
        idrid_df.to_csv(IDRID_XAI_METRICS_PATH, index=False)

        summary = {
            "evaluated_model": str(EVAL_MODEL_PATH),
            "model_variant": IDRID_MODEL_TAG,
            "benchmark": "IDRiD segmentation (external pixel-level lesion masks)",
            "lesion_codes": LESION_CODES_FOR_IOU,
            "explained_class": "model predicted class",
            "heatmap_top_percent": float(IDRID_TOP_PERCENT),
            "n_images_evaluated": int(len(idrid_df)),
            "pred_class_distribution": {
                CLASS_NAMES[c]: int((idrid_df["pred_class_id"] == c).sum())
                for c in range(NUM_CLASSES)
            },
            "pointing_game_accuracy": float(idrid_df["pointing_hit"].mean()),
            "mean_energy_inside_mask": float(idrid_df["energy_inside_mask"].mean()),
            f"mean_iou_top_{IDRID_TOP_PERCENT:g}pct": float(idrid_df["iou_top_percent"].mean()),
            f"mean_dice_top_{IDRID_TOP_PERCENT:g}pct": float(idrid_df["dice_top_percent"].mean()),
            f"mean_precision_top_{IDRID_TOP_PERCENT:g}pct": float(idrid_df["precision_top_percent"].mean()),
            f"mean_recall_top_{IDRID_TOP_PERCENT:g}pct": float(idrid_df["recall_top_percent"].mean()),
            "mean_iou_by_threshold": {
                f"top_{t:g}pct": float(idrid_df[f"iou_top_{t:g}pct"].mean())
                for t in IDRID_THRESHOLD_SWEEP
            },
        }

        with open(IDRID_XAI_SUMMARY_PATH, "w") as f:
            json.dump(summary, f, indent=4)

        print("Saved IDRiD Grad-CAM IoU metrics:", IDRID_XAI_METRICS_PATH)
        print("Saved IDRiD Grad-CAM IoU summary:", IDRID_XAI_SUMMARY_PATH)
        print(json.dumps(summary, indent=4))

        # ---- qualitative grid: image | Grad-CAM | lesion mask (GT) | overlay ----
        if panels:
            n = len(panels)
            fig, axes = plt.subplots(n, 4, figsize=(14, 3.4 * n))
            if n == 1:
                axes = np.expand_dims(axes, 0)
            for r, (img_rgb, hmap, mask, did, pname, iou) in enumerate(panels):
                axes[r, 0].imshow(img_rgb)
                axes[r, 0].set_title(f"{did}\npred={pname}")
                axes[r, 0].axis("off")
                axes[r, 1].imshow(hmap, cmap="jet")
                axes[r, 1].set_title("Grad-CAM")
                axes[r, 1].axis("off")
                axes[r, 2].imshow(mask, cmap="gray")
                axes[r, 2].set_title("Lesion mask (GT)")
                axes[r, 2].axis("off")
                axes[r, 3].imshow(img_rgb)
                axes[r, 3].imshow(hmap, cmap="jet", alpha=0.45)
                axes[r, 3].contour(mask, colors="lime", linewidths=1.0)
                axes[r, 3].set_title(f"overlay | IoU={iou:.3f}")
                axes[r, 3].axis("off")
            plt.tight_layout()
            plt.savefig(IDRID_GRID_PATH, dpi=180, bbox_inches="tight")
            plt.show()
            print("Saved IDRiD qualitative grid:", IDRID_GRID_PATH)

        display(idrid_df.head(10))
